In [1]:
import requests
import shutil
import csv
import os
import random
from tqdm import tqdm
import urllib

In [2]:
def download_images(csv_file, image_folder, number_of_images_per_class):
    with open(csv_file, 'r') as file:
        reader = csv.reader(file)
        next(reader)  # Skip header row

        rows = list(reader)
        if number_of_images_per_class > len(rows):
            print(f"Requested {number_of_images_per_class} images, but only {len(rows)} are available. Downloading all available images.")
            number_of_images_per_class = len(rows)
        random_rows = random.sample(rows, number_of_images_per_class)

        for row in tqdm(random_rows):
            image_url = row[1]
            extension = os.path.splitext(urllib.parse.urlparse(image_url).path)[1]
            filename = row[0] + extension
            file_path = os.path.join(image_folder, filename)

            try:
                response = requests.get(image_url, stream=True)
                response.raise_for_status()  # Check if the request was successful

                with open(file_path, 'wb') as out_file:
                    shutil.copyfileobj(response.raw, out_file)

            except requests.exceptions.RequestException as e:
                print(f"Failed to download {image_url}: {e}")

In [3]:
root_dir = "./"
csv_files = os.path.join(root_dir, "csv_files/")
os.makedirs(os.path.join(root_dir, "images/"), exist_ok=True) # Create images directory if it doesn't exist
image_folders = os.path.join(root_dir, "images")

In [4]:
number_of_images_per_class = 350

for each_csv in os.listdir(csv_files):
    folder_name = each_csv.split(".")[0]
    # Create a folder for each class
    os.makedirs(os.path.join(image_folders, folder_name), exist_ok=True)
    csv_file = os.path.join(csv_files, each_csv)
    image_folder = os.path.join(image_folders, folder_name)
    download_images(csv_file, image_folder, number_of_images_per_class)

100%|██████████| 350/350 [16:37<00:00,  2.85s/it]


In [5]:
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" #show only error messages, to avoid printing a tonne of verbose warnings!

import tensorflow as tf
tf.get_logger().setLevel('ERROR') #set TensorFlow logger level to ERROR 
print("tensorflow: ",tf.__version__)

import keras
print("keras: ",keras.__version__)

import keras_hub
print("keras hub: ",keras_hub.__version__)

import keras_cv
print("keras cv: ",keras_cv.__version__)

tensorflow:  2.21.0
keras:  3.14.1
keras hub:  0.29.0
keras cv:  0.9.0


In [ ]:
print(keras_hub.models.ImageObjectDetector.presets.keys())

In [ ]:
retinanet = keras_hub.models.RetinaNetObjectDetector.from_preset(
    "retinanet_resnet50_fpn_coco"
)

In [ ]:
image_size=(800, 800)
batch_size=8

image_dir = "./images/"

tf_dataset = keras.utils.image_dataset_from_directory(
    image_dir,
    image_size=image_size,
    batch_size=batch_size,
    pad_to_aspect_ratio=True
)